# P11 — Dermatology Image Classifier with Patient Report NLP
## EDA & End-to-End Pipeline Demo

**Pipeline Overview:**
1. HAM10000 Dataset EDA
2. CNN Training (ResNet-18 transfer learning)
3. Grad-CAM visualization
4. NLP: NER entity extraction
5. Severity classification (DistilBERT)
6. NLG patient report generation
7. BLEU evaluation

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.utils.config import *
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## 1. HAM10000 EDA

In [ ]:
# Load metadata
try:
    df = pd.read_csv(HAM_CSV)
    print(f'Dataset shape: {df.shape}')
    print(df.head())
except FileNotFoundError:
    print('HAM10000 not downloaded yet.')
    print('Run: kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection')
    df = None

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Class distribution
    counts = df['dx'].value_counts()
    axes[0].bar(counts.index, counts.values, color=plt.cm.Set2.colors)
    axes[0].set_title('HAM10000 Class Distribution', fontsize=13)
    axes[0].set_xlabel('Diagnosis Class')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=30)

    # Age distribution by class
    df.boxplot(column='age', by='dx', ax=axes[1])
    axes[1].set_title('Age Distribution by Class')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 2. CNN Training

In [ ]:
from src.dl.model import build_model
model = build_model()
print(model)

In [ ]:
# To train:
# !python ../src/dl/train.py --epochs 15 --batch_size 32 --lr 0.01

# Load training history if already trained
import json
history_path = os.path.join(OUTPUT_DIR, 'training_history.json')
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    epochs = range(1, len(history['train']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [e['loss'] for e in history['train']], label='Train')
    axes[0].plot(epochs, [e['loss'] for e in history['val']], label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(epochs, [e['macro_f1'] for e in history['train']], label='Train')
    axes[1].plot(epochs, [e['macro_f1'] for e in history['val']], label='Val')
    axes[1].set_title('Macro-F1'); axes[1].legend()
    plt.suptitle('Training Curves')
    plt.tight_layout()
    plt.show()
else:
    print('No training history found. Train the model first.')

## 3. Grad-CAM Visualization

In [ ]:
# Example Grad-CAM (replace with real image path)
# from src.dl.gradcam import visualize_gradcam
# from src.dl.model import load_checkpoint
#
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = build_model().to(device)
# load_checkpoint(os.path.join(MODEL_DIR, 'best_model.pt'), model)
#
# result = visualize_gradcam('data/example.jpg', model, device)
# from IPython.display import Image
# Image(result['overlay_path'])
print('Uncomment and run with a real dermoscopic image.')

## 4. NLP Pipeline — NER

In [ ]:
from src.nlp.ner_pipeline import build_ner_pipeline, extract_entities

nlp = build_ner_pipeline()

sample_reports = [
    "I've had a dark brown mole on my left forearm for 3 months.",
    "There is a melanoma on my upper back that has grown for 2 weeks.",
    "Patient reports a basal cell carcinoma on the scalp since childhood.",
]

for report in sample_reports:
    ents = extract_entities(report, nlp)
    print(f'Report: {report[:60]}...')
    print(f'  Entities: {ents}\n')

## 5. Synthetic Dataset EDA

In [ ]:
from src.nlp.data_gen import generate_synthetic_dataset

df_syn = generate_synthetic_dataset(100)
print(df_syn[['report_id', 'raw_report', 'severity']].head(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_syn['severity'].value_counts().plot.bar(ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336'])
axes[0].set_title('Severity Distribution')

df_syn['lesion_type'].value_counts().head(7).plot.barh(ax=axes[1], color='steelblue')
axes[1].set_title('Top Lesion Types')
plt.tight_layout()
plt.show()

## 6. NLG Report Generation

In [ ]:
from src.nlp.nlg_module import PatientReportGenerator

generator = PatientReportGenerator()
entities = {
    'body_part': ['left forearm'],
    'lesion_type': ['melanoma'],
    'symptom_duration': ['3 months'],
}

report = generator.generate(
    cnn_class='mel',
    cnn_confidence=0.812,
    entities=entities,
    severity='severe',
    report_id='DEMO-NB-001',
)
print(report)

## 7. BLEU Evaluation

In [ ]:
from src.nlp.nlg_module import compute_bleu

# Generate reports for all synthetic data
generated_reports = []
for _, row in df_syn.iterrows():
    ents = extract_entities(row['raw_report'], nlp)
    rep = generator.generate(
        cnn_class='nv',
        cnn_confidence=0.80,
        entities=ents,
        severity=row['severity'],
        report_id=row['report_id'],
    )
    generated_reports.append(rep)

references = df_syn['reference_report'].tolist()
bleu = compute_bleu(generated_reports, references)
print(f'Corpus BLEU Score: {bleu:.4f}')